# CluStream — Clustering Evolving Data Streams

> Aggarwal, C. C. et al. (2003) *"A Framework for Clustering Evolving Data Streams"* — VLDB 2003

---

## Motivation

| Challenge | Limitation of static algorithms |
|---|---|
| Infinite, high-speed stream | Cannot store all points |
| Concept drift / evolving clusters | Batch k-means uses a fixed snapshot |
| Time-window queries | "What were the clusters in the last hour?" is unanswerable |

CluStream solves all three with a **two-component framework**:

1. **Online micro-clustering** — compact sufficient statistics updated incrementally.
2. **Offline macro-clustering** — weighted k-means on stored micro-cluster *snapshots*.

---

## Micro-Cluster Sufficient Statistics

A micro-cluster $MC$ for $n$ points $\{x_1,\ldots,x_n\}$ arriving at times $\{t_1,\ldots,t_n\}$ is:

$$MC = \left(\overrightarrow{CF2^x},\; \overrightarrow{CF1^x},\; CF2^t,\; CF1^t,\; n\right)$$

| Statistic | Definition | Role |
|---|---|---|
| $\overrightarrow{CF2^x}$ | $\sum_{i=1}^n x_i^2$ (element-wise) | Compute variance / radius |
| $\overrightarrow{CF1^x}$ | $\sum_{i=1}^n x_i$ | Compute centroid |
| $CF2^t$ | $\sum_{i=1}^n t_i^2$ | Temporal spread |
| $CF1^t$ | $\sum_{i=1}^n t_i$ | Mean arrival time |
| $n$ | Point count | Weight |

### Derived quantities

$$\text{centroid} = \frac{\overrightarrow{CF1^x}}{n}, \qquad
\text{radius} = \sqrt{\frac{\overrightarrow{CF2^x}}{n} - \left(\frac{\overrightarrow{CF1^x}}{n}\right)^2}$$

These are **additive**: merging two micro-clusters requires only element-wise addition of their statistics — $O(d)$ per update.

---

## Online Phase — Point Assignment

For each arriving point $p$:

1. Find the nearest micro-cluster $MC^*$ by centroid distance.
2. Compute the **maximum boundary radius** $\text{RMSD}(MC^*)$:

$$\text{RMSD} = \sqrt{\frac{CF2^x}{n} - \left(\frac{CF1^x}{n}\right)^2}$$

3. If $\|p - \text{centroid}(MC^*)\| \leq t \cdot \text{RMSD}(MC^*)$ (with $t$ a threshold, default $t=2$): **absorb** $p$ into $MC^*$.
4. Otherwise: **create** a new micro-cluster — if the micro-cluster count $q$ is exceeded, either:
   - Delete the micro-cluster with the oldest mean arrival time (stalest), **or**
   - Merge the two closest micro-clusters.

---

## Pyramidal Time Frame — Snapshot Storage

CluStream stores periodic *snapshots* of all micro-cluster statistics using a **pyramidal time frame**:

- Order-$\ell$ snapshots are kept every $\alpha^\ell$ time units, retaining the last $l$ snapshots at each order.
- This yields $O(\log T)$ total snapshots for a stream of length $T$.
- A time-window query $[t_1, t_2]$ is answered by **differencing** two snapshots:

$$CF2^x_{[t_1,t_2]} = CF2^x_{t_2} - CF2^x_{t_1}$$

---

## Offline Phase — Macro-Clustering

Weighted k-means on the $q$ micro-cluster centroids, using $n$ as the weight. This produces the final $k$ macro-clusters for any stored time window.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports OK')

---
## 1  Micro-Cluster Class

In [ ]:
class MicroCluster:
    """CluStream micro-cluster sufficient statistics."""

    def __init__(self, point: np.ndarray, timestamp: float):
        self.cf2x = point ** 2          # sum of squared feature values
        self.cf1x = point.copy()        # sum of feature values
        self.cf2t = timestamp ** 2      # sum of squared timestamps
        self.cf1t = timestamp           # sum of timestamps
        self.n    = 1                   # point count

    # ------------------------------------------------------------------ #
    @property
    def centroid(self) -> np.ndarray:
        return self.cf1x / self.n

    @property
    def radius(self) -> float:
        """Root Mean Square Deviation (RMSD) — per-dimension averaged."""
        variance = self.cf2x / self.n - (self.cf1x / self.n) ** 2
        # clip numerical noise
        return float(np.sqrt(np.maximum(variance, 0).mean()))

    @property
    def mean_time(self) -> float:
        return self.cf1t / self.n

    # ------------------------------------------------------------------ #
    def insert(self, point: np.ndarray, timestamp: float):
        """Absorb a new point — O(d) additive update."""
        self.cf2x += point ** 2
        self.cf1x += point
        self.cf2t += timestamp ** 2
        self.cf1t += timestamp
        self.n    += 1

    def merge(self, other: 'MicroCluster'):
        """Merge another micro-cluster into this one."""
        self.cf2x += other.cf2x
        self.cf1x += other.cf1x
        self.cf2t += other.cf2t
        self.cf1t += other.cf1t
        self.n    += other.n

    def snapshot(self) -> dict:
        """Return a copy of the current sufficient statistics."""
        return dict(cf2x=self.cf2x.copy(), cf1x=self.cf1x.copy(),
                    cf2t=self.cf2t, cf1t=self.cf1t, n=self.n)


# Quick sanity check
rng = np.random.default_rng(0)
mc = MicroCluster(np.array([1.0, 2.0]), timestamp=1)
mc.insert(np.array([3.0, 4.0]), timestamp=2)
print(f'centroid : {mc.centroid}')   # [2, 3]
print(f'radius   : {mc.radius:.4f}')
print(f'n        : {mc.n}')

---
## 2  CluStream Online Component

In [ ]:
class CluStreamOnline:
    """
    Online phase of CluStream.

    Parameters
    ----------
    q          : maximum number of micro-clusters to maintain
    t          : boundary threshold multiplier (default 2)
    init_size  : number of points used for k-means initialisation
    """

    def __init__(self, q: int = 50, t: float = 2.0, init_size: int = 500):
        self.q         = q
        self.t         = t
        self.init_size = init_size
        self.mcs: list[MicroCluster] = []
        self._buffer: list[tuple] = []   # (point, timestamp) during init
        self._initialized = False
        self.timestamp = 0

    # ------------------------------------------------------------------ #
    def _init_from_buffer(self):
        """Bootstrap q micro-clusters with k-means on the initial buffer."""
        points = np.array([p for p, _ in self._buffer])
        km = KMeans(n_clusters=self.q, n_init=3, random_state=RANDOM_STATE)
        labels = km.fit_predict(points)
        for k in range(self.q):
            idx = np.where(labels == k)[0]
            if len(idx) == 0:
                continue
            pts_k = [self._buffer[i] for i in idx]
            mc = MicroCluster(pts_k[0][0], pts_k[0][1])
            for p, ts in pts_k[1:]:
                mc.insert(p, ts)
            self.mcs.append(mc)
        self._initialized = True

    # ------------------------------------------------------------------ #
    def update(self, point: np.ndarray):
        """Process a single incoming stream point."""
        self.timestamp += 1

        # --- initialisation phase ---
        if not self._initialized:
            self._buffer.append((point.copy(), float(self.timestamp)))
            if len(self._buffer) >= self.init_size:
                self._init_from_buffer()
            return

        # --- find nearest micro-cluster ---
        centroids = np.array([mc.centroid for mc in self.mcs])
        dists = np.linalg.norm(centroids - point, axis=1)
        nearest_idx = int(np.argmin(dists))
        mc_nearest = self.mcs[nearest_idx]

        # compute dynamic boundary
        rmsd = mc_nearest.radius
        boundary = self.t * rmsd if rmsd > 0 else self.t * 0.1  # fallback for singleton

        if dists[nearest_idx] <= boundary:
            mc_nearest.insert(point, float(self.timestamp))
        else:
            # Create new micro-cluster; free a slot if needed
            self._make_room()
            self.mcs.append(MicroCluster(point, float(self.timestamp)))

    def _make_room(self):
        """Free one slot: delete the stalest MC, or merge the closest pair."""
        if len(self.mcs) < self.q:
            return
        # Delete the micro-cluster with oldest mean arrival time
        stalest = int(np.argmin([mc.mean_time for mc in self.mcs]))
        self.mcs.pop(stalest)

    # ------------------------------------------------------------------ #
    @property
    def micro_centroids(self) -> np.ndarray:
        return np.array([mc.centroid for mc in self.mcs])

    @property
    def micro_weights(self) -> np.ndarray:
        return np.array([mc.n for mc in self.mcs], dtype=float)


print('CluStreamOnline class defined')

---
## 3  Offline Macro-Clustering

In [ ]:
def macro_cluster(online: CluStreamOnline, k: int) -> np.ndarray:
    """
    Run weighted k-means on the micro-cluster centroids.

    Returns
    -------
    labels : int array of shape (n_micro_clusters,)
    """
    centroids = online.micro_centroids
    weights   = online.micro_weights

    # sklearn KMeans accepts sample_weight
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(centroids, sample_weight=weights)
    return labels, km.cluster_centers_


print('macro_cluster() defined')

---
## 4  Simulated Evolving Stream

In [ ]:
# ── Stream parameters ──────────────────────────────────────────────────────
N_TOTAL    = 5_000   # total stream length
N_CLUSTERS = 4       # true number of macro-clusters
SHIFT_AT   = 2_500   # cluster centres shift at this time step

rng = np.random.default_rng(RANDOM_STATE)

# Phase 1 centres
centres_p1 = np.array([[2,2],[6,2],[6,6],[2,6]], dtype=float)
# Phase 2 centres (translate clusters 0 and 2 to simulate drift)
centres_p2 = centres_p1.copy()
centres_p2[0] += [3, 3]
centres_p2[2] -= [3, 3]

stream_X, stream_y = [], []
for i in range(N_TOTAL):
    centres = centres_p1 if i < SHIFT_AT else centres_p2
    c = rng.integers(0, N_CLUSTERS)
    point = rng.normal(centres[c], 0.6)
    stream_X.append(point)
    stream_y.append(c)

stream_X = np.array(stream_X)
stream_y = np.array(stream_y)

print(f'Stream: {N_TOTAL:,} points, drift at t={SHIFT_AT:,}')
print(f'Shape : {stream_X.shape}')

---
## 5  Run CluStream

In [ ]:
cs = CluStreamOnline(q=50, t=2.0, init_size=500)

for x in stream_X:
    cs.update(x)

print(f'Active micro-clusters : {len(cs.mcs)}')
print(f'Total weight (points) : {cs.micro_weights.sum():.0f}')

In [ ]:
# ── Macro-cluster the micro-clusters into k=4 groups ──────────────────────
mc_labels, macro_centres = macro_cluster(cs, k=N_CLUSTERS)

# Map each original stream point through its nearest micro-cluster → macro label
centroids_arr = cs.micro_centroids
all_dists = np.linalg.norm(stream_X[:, None, :] - centroids_arr[None, :, :], axis=2)
nearest_mc = np.argmin(all_dists, axis=1)
pred_labels = mc_labels[nearest_mc]

ari = adjusted_rand_score(stream_y, pred_labels)
print(f'Adjusted Rand Index (full stream): {ari:.4f}')

---
## 6  Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
palette = np.array(COLORS)

# ── (A) Raw stream (first 1000 pts) ───────────────────────────────────────
ax = axes[0]
sample = np.random.choice(len(stream_X), size=1000, replace=False)
ax.scatter(stream_X[sample, 0], stream_X[sample, 1],
           c=[COLORS[l] for l in stream_y[sample]], alpha=0.5, s=10)
ax.set_title('(A) Raw stream (1 000-pt sample, true labels)', fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('y')

# ── (B) Micro-clusters ────────────────────────────────────────────────────
ax = axes[1]
mc_c = cs.micro_centroids
mc_w = cs.micro_weights
ax.scatter(mc_c[:, 0], mc_c[:, 1],
           s=mc_w / mc_w.max() * 400,
           c=[COLORS[l % 4] for l in mc_labels],
           alpha=0.8, edgecolors='k', linewidths=0.5)
ax.scatter(macro_centres[:, 0], macro_centres[:, 1],
           marker='*', s=350, c='black', zorder=5, label='Macro centres')
ax.set_title('(B) Micro-clusters (size ∝ weight, colour = macro label)', fontsize=11)
ax.set_xlabel('x'); ax.legend(loc='upper right')

# ── (C) CluStream predictions on stream sample ────────────────────────────
ax = axes[2]
ax.scatter(stream_X[sample, 0], stream_X[sample, 1],
           c=[COLORS[l % 4] for l in pred_labels[sample]], alpha=0.5, s=10)
ax.scatter(macro_centres[:, 0], macro_centres[:, 1],
           marker='*', s=350, c='black', zorder=5)
ax.set_title(f'(C) CluStream predictions  (ARI = {ari:.3f})', fontsize=11)
ax.set_xlabel('x')

plt.suptitle('CluStream on Evolving 2-D Stream', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 7  Pyramidal Time Frame (Snapshot Storage)

In [ ]:
class PyramidalSnapshots:
    """
    Stores micro-cluster snapshots using a pyramidal time frame.

    Order-l snapshots are kept every alpha^l time units;
    at most `l` snapshots are retained per order.

    Parameters
    ----------
    alpha : base of the pyramid (default 2)
    l     : number of snapshots per order (default 3)
    max_order : highest order to maintain (default 8  → 2^8=256 time-unit horizon)
    """

    def __init__(self, alpha: int = 2, l: int = 3, max_order: int = 8):
        self.alpha     = alpha
        self.l         = l
        self.max_order = max_order
        # dict: order -> deque of (timestamp, snapshot_list)
        from collections import deque
        self._store = {o: deque(maxlen=l) for o in range(max_order + 1)}

    def record(self, t: int, online: CluStreamOnline):
        """Decide which orders need a snapshot at time t and store them."""
        snap = [mc.snapshot() for mc in online.mcs]
        for o in range(self.max_order + 1):
            if t % (self.alpha ** o) == 0:
                self._store[o].append((t, snap))

    def orders_summary(self) -> None:
        print(f'{'Order':>6}  {'Interval':>10}  {'Snapshots stored':>16}')
        print('-' * 38)
        for o in range(self.max_order + 1):
            if self._store[o]:
                times = [t for t, _ in self._store[o]]
                print(f'{o:>6}  {self.alpha**o:>10}  {str(times):>16}')


# ── Replay stream with snapshot recording ─────────────────────────────────
cs2 = CluStreamOnline(q=50, t=2.0, init_size=500)
pyramid = PyramidalSnapshots(alpha=2, l=3, max_order=8)

for i, x in enumerate(stream_X, start=1):
    cs2.update(x)
    if cs2._initialized:
        pyramid.record(i, cs2)

print('Snapshots stored per order:')
pyramid.orders_summary()

---
## 8  Time-Window Query — Before vs After Drift

In [ ]:
# Compare macro-clustering in two time windows:
#   window_early  — first 2500 points (pre-drift)
#   window_late   — last 2500 points (post-drift)
#
# We approximate by re-running CluStream on each half separately.
# (A full implementation would use snapshot differencing.)

def run_clustream(data, q=50, init_size=500):
    cs = CluStreamOnline(q=q, t=2.0, init_size=init_size)
    for x in data:
        cs.update(x)
    return cs

cs_early = run_clustream(stream_X[:SHIFT_AT])
cs_late  = run_clustream(stream_X[SHIFT_AT:])

labels_early, centres_early = macro_cluster(cs_early, k=N_CLUSTERS)
labels_late,  centres_late  = macro_cluster(cs_late,  k=N_CLUSTERS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cs_w, ctrs, title in [
    (axes[0], cs_early, centres_early, 'Window 1 (pre-drift  t=0..2500)'),
    (axes[1], cs_late,  centres_late,  'Window 2 (post-drift t=2500..5000)'),
]:
    mc_c = cs_w.micro_centroids
    mc_w = cs_w.micro_weights
    mc_l, _ = macro_cluster(cs_w, k=N_CLUSTERS)
    ax.scatter(mc_c[:, 0], mc_c[:, 1],
               s=mc_w / mc_w.max() * 300,
               c=[COLORS[l % 4] for l in mc_l],
               alpha=0.8, edgecolors='k', linewidths=0.4)
    ax.scatter(ctrs[:, 0], ctrs[:, 1],
               marker='*', s=300, c='black', zorder=5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('CluStream — Cluster Structure Before and After Drift', fontsize=13)
plt.tight_layout()
plt.show()

print('Centre shift (cluster 0):', np.linalg.norm(centres_p2[0] - centres_p1[0]))

---
## 9  Effect of q (Number of Micro-Clusters) on ARI

In [ ]:
q_values = [10, 20, 30, 50, 75, 100]
ari_results = []

for q in q_values:
    cs_q = CluStreamOnline(q=q, t=2.0, init_size=min(500, q * 5))
    for x in stream_X:
        cs_q.update(x)
    mc_l, _ = macro_cluster(cs_q, k=N_CLUSTERS)
    # Map stream points → predictions
    mc_c = cs_q.micro_centroids
    d = np.linalg.norm(stream_X[:, None, :] - mc_c[None, :, :], axis=2)
    pred = mc_l[np.argmin(d, axis=1)]
    ari_results.append(adjusted_rand_score(stream_y, pred))
    print(f'q={q:>3}  ARI={ari_results[-1]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(q_values, ari_results, 'o-', color='steelblue', linewidth=2)
plt.xlabel('Number of micro-clusters  q', fontsize=12)
plt.ylabel('Adjusted Rand Index', fontsize=12)
plt.title('CluStream: ARI vs q  (k=4, 5 000-pt stream)', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10  CluStream vs Batch K-Means

In [ ]:
# Batch k-means on the full stream (impractical at scale, used as reference)
km_batch = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE)
labels_batch = km_batch.fit_predict(stream_X)
ari_batch = adjusted_rand_score(stream_y, labels_batch)

print(f'Batch k-means ARI  : {ari_batch:.4f}  (needs all {N_TOTAL:,} pts in memory)')
print(f'CluStream ARI      : {ari:.4f}  (O(q·d) memory, q={cs.q})')
print()
print('Memory comparison:')
print(f'  Batch k-means : {stream_X.nbytes / 1024:.1f} KB  for {N_TOTAL:,} points')
mc_mem = len(cs.mcs) * (4 * stream_X.shape[1] + 2) * 8  # rough bytes
print(f'  CluStream     : ~{mc_mem} bytes  for {len(cs.mcs)} micro-clusters')

---
## 11  CluStream vs DenStream — Side-by-Side

In [ ]:
comparison = {
    'Property':          ['Micro-cluster model', 'Weighting scheme',
                          'Offline phase',       'Time windows',
                          'Memory model',        'Handles outliers',
                          'Concept drift'],
    'CluStream':         ['CF sufficient stats', 'Uniform (count)',
                          'Weighted k-means',    'Yes (pyramidal frame)',
                          'Fixed q MCs',         'Implicit (staleness)',
                          'Via snapshot diff.'],
    'DenStream':         ['Fading CF stats', 'Exponential fading (λ)',
                          'DBSCAN on p-MCs', 'No',
                          'Dynamic p-MC/o-MC', 'Explicit (o-MC pruning)',
                          'Via fading function'],
}

import pandas as pd
df = pd.DataFrame(comparison).set_index('Property')
print(df.to_string())

---
## 12  Key Take-Aways

| Concept | Summary |
|---|---|
| **Sufficient statistics** | $\overrightarrow{CF2^x}, \overrightarrow{CF1^x}, CF2^t, CF1^t, n$ — $O(d)$ per update, additive under merge |
| **Online assignment** | Nearest micro-cluster within $t \cdot \text{RMSD}$; else create new MC |
| **Slot management** | Delete stalest (oldest mean time) MC to keep count $\leq q$ |
| **Pyramidal snapshots** | $O(\log T)$ stored states enable arbitrary time-window queries |
| **Offline clustering** | Weighted k-means on $q$ centroids — decoupled from online speed |
| **Memory** | $O(q \cdot d)$ — independent of stream length $T$ |

---

### References

- Aggarwal, C. C., Han, J., Wang, J., & Yu, P. S. (2003). **A Framework for Clustering Evolving Data Streams**. *VLDB 2003*, 81–92.
- Cao, F. et al. (2006). **DenStream: A Clustering Algorithm over an Evolving Data Stream**. *SDM 2006*.
- Bifet, A. et al. (2018). **Machine Learning for Data Streams**. MIT Press.